# Day 15 — Naming outputs: alias, name.suffix, name.map

*100 Days of Polars · Phase 2: Expressions*

- **Post:** [Naming Outputs: alias, name.suffix and name.map](https://python.plainenglish.io/day-15-naming-outputs-alias-name-suffix-and-name-map-113f1a80283f)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Every output column needs a unique name. Polars picks one for you (the **leftmost** column in the expression); you override it with `alias`, keyword arguments, or the `name` namespace.

In [2]:
items = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
products = pl.read_csv("data/raw/olist_products_dataset.csv")

## 1. The default name

In [3]:
items.select(
    pl.col("freight_value") / pl.col("price"),   # named after the leftmost column
    pl.lit(1) + pl.col("order_item_id"),          # the leftmost thing is a literal
).head(3)

freight_value,literal
f64,i64
0.225637,2
0.083076,2
0.089799,2


## 2. `alias` and keyword arguments

In [4]:
items.select(
    (pl.col("freight_value") / pl.col("price")).alias("freight_ratio"),
    order_item_plus_one=pl.lit(1) + pl.col("order_item_id"),  # same thing, keyword style
).head(3)

freight_ratio,order_item_plus_one
f64,i64
0.225637,2
0.083076,2
0.089799,2


## 3. Many outputs, one naming rule: `name.suffix` / `name.prefix`

In [5]:
items.select(
    pl.col("price", "freight_value").sum().name.suffix("_sum"),
    pl.col("price", "freight_value").mean().name.suffix("_mean"),
)

price_sum,freight_value_sum,price_mean,freight_value_mean
f64,f64,f64,f64
1.3592e7,2.2519e6,120.653739,19.99032


In [6]:
items.group_by("seller_id").agg(
    pl.col("price", "freight_value").sum().name.prefix("total_"),
    pl.col("price", "freight_value").max().name.prefix("max_"),
).head(3)

seller_id,total_price,total_freight_value,max_price,max_freight_value
str,f64,f64,f64,f64
"""718539d38d07dd351c76db862760e2…",1318.9,659.05,99.9,34.91
"""5cf1ec37aea089f45110c70653a166…",50.0,43.1,20.0,20.8
"""8c351ed7c326c62123318f4f3246e9…",114.9,35.76,59.9,19.61


## 4. `name.map`: any Python function on the name

In [7]:
# Fix the Olist typo 'lenght' -> 'length' everywhere, in one expression
products.select(pl.all().name.map(lambda c: c.replace("lenght", "length"))).columns

['product_id',
 'product_category_name',
 'product_name_length',
 'product_description_length',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

In [8]:
# For a whole frame, rename also accepts a function
products.rename(lambda c: c.removeprefix("product_")).columns

['id',
 'category_name',
 'name_lenght',
 'description_lenght',
 'photos_qty',
 'weight_g',
 'length_cm',
 'height_cm',
 'width_cm']

In [9]:
products.select(pl.col("^product_.*_cm$").name.to_uppercase()).columns

['PRODUCT_LENGTH_CM', 'PRODUCT_HEIGHT_CM', 'PRODUCT_WIDTH_CM']

## 5. `name.keep`: back to the root name

In [10]:
items.select((pl.col("price") * 1.1).alias("with_tax").name.keep()).columns

['price']

**What does `name.*` build on?** `name.keep()` goes back to the **root** column. The others (`suffix`, `prefix`, `map`) build on the **current** name, so after an `alias` they extend the alias. (The one-line docstring summaries in 2.0.0 still say "root column name"; the alias handling was corrected in the docs by [pola-rs/polars#23773](https://github.com/pola-rs/polars/pull/23773), merged July 2025.)

In [11]:
print(pl.col("price").alias("cost").name.suffix("_x").meta.output_name())   # cost_x: current name, not root

# name.keep can't be resolved without a frame: meta.output_name() raises, so run it in a select
try:
    pl.col("price").alias("cost").name.keep().meta.output_name()
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])
print(items.select(pl.col("price").alias("cost").name.keep()).columns)      # ['price']

cost_x
ComputeError -> not yet implemented: `name.keep` is not allowed here
['price']


## 6. Gotcha: same name in `with_columns` silently replaces

In `select` a duplicate name raises. In `with_columns` an output that matches an existing column **overwrites** it, with no warning.

In [12]:
original = items["price"].sum()
replaced = items.with_columns(pl.col("price") * 1.1)  # output is named 'price' -> overwrites
print(round(original, 2), round(replaced["price"].sum(), 2))

13591643.7 14950808.07


In [13]:
# Sometimes that is what you want (cleaning in place). If not, name the output.
items.with_columns(price_with_tax=pl.col("price") * 1.1).select("price", "price_with_tax").head(3)

price,price_with_tax
f64,f64
58.9,64.79
239.9,263.89
199.0,218.9


**Replace or new column? Only the final name decides.** It does not matter whether the name came from the default, a keyword, `alias` or `name.*`.

In [14]:
checks = {
    "default name (price)":       items.with_columns(pl.col("price") * 1.1),
    "keyword, new name":          items.with_columns(price_tax=pl.col("price") * 1.1),
    "alias to an existing name":  items.with_columns((pl.col("price") * 1.1).alias("price")),
    "name.suffix, new names":     items.with_columns(pl.col("price", "freight_value").round(0).name.suffix("_r")),
}
for label, df in checks.items():
    print(f"{label:<28} -> {df.width} columns, price sum {df['price'].sum():,.2f}")

default name (price)         -> 7 columns, price sum 14,950,808.07
keyword, new name            -> 8 columns, price sum 13,591,643.70
alias to an existing name    -> 7 columns, price sum 14,950,808.07
name.suffix, new names       -> 9 columns, price sum 13,591,643.70


**`name.*` cannot separate two outputs built from the same column.** The suffix has to describe the operation.

In [15]:
try:
    items.select(pl.col("price").sum().name.suffix("_x"), pl.col("price").mean().name.suffix("_x"))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

items.select(pl.col("price").sum().name.suffix("_sum"), pl.col("price").mean().name.suffix("_mean"))

DuplicateError -> projections contained duplicate output name 'price_x'. It's possible that multiple expressions are returning the same default column name. If this is the case, try renaming the columns with `.alias("new_name")` to avoid duplicate column names.


price_sum,price_mean
f64,f64
1.3592e7,120.653739


## Try it yourself

1. Fix yesterday's exercise: min and max of every numeric `products` column in one `select`, using suffixes.
2. Use `name.map` to turn every `products` column into Title Case without the `product_` prefix.
3. What does `(pl.col("price") + pl.col("freight_value")).alias("x").meta.output_name()` return? And with `.name.keep()` added?